In [215]:
""" 
Class 10: Model Evaluation and Bias-Variance Tradeoff

Objective:
    - Implement a robust linear regression model for house price prediction.
    - Use proper data splitting (train/validation/test).
    - Apply cross-validation for reliable performance estimation.
    - Use scikit-learn's train_test_split for reproducibility.
    - Standardize features and encode categorical variables.
    - Evaluate using MSE and KL-divergence.
    - Visualize bias-variance tradeoff via learning curves.  
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import entropy
import os

%matplotlib notebook
plt.style.use('E:\\PyCharmProjects\\pythonProject\\test\\deeplearning.mplstyle')

In [216]:
"""
Machine Learning is a subset of artificial intelligence where algorithms learn patterns from data to make predictions or decisions without being explicitly programmed for each task.

Machine Learning Tasks:
1. Supervised Learning: Learns from labeled data
   1.1. Regression: Predicts a numerical value i.e. house prediction
   1.2. Classification: Predicts the class label i.e. classify image to dogs / cats
2. Unsupervised Learning: Finds patterns in unlabeled data.
3. Reinforcement Learning: Learns by interacting with an environment to maximize a reward.

RQ: How does the bias-variance tradeoff in linear regression for house price prediction change as training data size increases, and what is the optimal data threshold for minimizing generalization error?
""";

In [217]:
"""
THEORETICAL CONCEPT: Dataset & Task Definition
- Task: Regression (predict continuous value: house price)
- Input (X): Multiple features (area, bedrooms, etc.)
- Output (y): 'price' (target variable)
- Model: Linear Regression → y = Xw + b
"""

# Define paths
ROOT_DIR = "E:\\PyCharmProjects\\pythonProject"
DATA_DIR = os.path.join(ROOT_DIR, "data")
DATASET_PATH = os.path.join(DATA_DIR, "housing.csv")

In [218]:
# Load dataset
housing_dataset = pd.read_csv(DATASET_PATH)
housing_dataset.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [219]:
"""
THEORETICAL CONCEPT: Feature Selection
- Remove irrelevant or redundant columns.
- Keep only predictive features and target.
"""
housing_dataset = housing_dataset[[
    'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
    'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
    'parking', 'prefarea', 'furnishingstatus', 'price'
]]

print("Selected columns:", housing_dataset.columns.tolist())

Selected columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus', 'price']


In [220]:
"""
THEORETICAL CONCEPT: Data Cleaning & Preprocessing
1. Handle missing values (if any).
2. Standardize numerical features → zero mean, unit variance.
   → Helps gradient descent converge faster.
3. Encode categorical variables → convert to numerical codes.
"""

# Check for missing values
print("Missing values:\n", housing_dataset.isnull().sum())

Missing values:
 area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
price               0
dtype: int64


In [221]:
housing_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   area              545 non-null    int64 
 1   bedrooms          545 non-null    int64 
 2   bathrooms         545 non-null    int64 
 3   stories           545 non-null    int64 
 4   mainroad          545 non-null    object
 5   guestroom         545 non-null    object
 6   basement          545 non-null    object
 7   hotwaterheating   545 non-null    object
 8   airconditioning   545 non-null    object
 9   parking           545 non-null    int64 
 10  prefarea          545 non-null    object
 11  furnishingstatus  545 non-null    object
 12  price             545 non-null    int64 
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [222]:
# Separate numerical and categorical columns
numerical_cols = housing_dataset.select_dtypes(include='number').columns.drop('price')
categorical_cols = housing_dataset.select_dtypes(include='object').columns

print("Numerical columns:", numerical_cols.tolist())
print("Categorical columns:", categorical_cols.tolist())

Numerical columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Categorical columns: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']


In [223]:
"""
def __init__(self,
             *,
             copy: Any = True,
             with_mean: Any = True,
             with_std: Any = True) -> None
             
Standardize features by removing the mean and scaling to unit variance.
The standard score of a sample x is calculated as:
z = (x - u) / s
where u is the mean of the training samples or zero if with_mean=False, and s is the standard deviation of the training samples or one if with_std=False.
"""
# Standardize numerical features

scaler = StandardScaler()
housing_dataset[numerical_cols] = scaler.fit_transform(
    housing_dataset[numerical_cols]
)

housing_dataset[numerical_cols].head()

,area,bedrooms,bathrooms,stories,parking
0,1.046726,1.403419,1.421812,1.378217,1.517692
1,1.757010,1.403419,5.405809,2.532024,2.679409
2,2.218232,0.047278,1.421812,0.224410,1.517692
3,1.083624,1.403419,1.421812,0.224410,2.679409
4,1.046726,1.403419,-0.570187,0.224410,1.517692


In [224]:
# Encode categorical features using label encoding (via pandas Categorical codes)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in categorical_cols:
    housing_dataset[col] = le.fit_transform(housing_dataset[col])

housing_dataset.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price
0,1.046726,1.403419,1.421812,1.378217,1,0,0,0,1,1.517692,1,0,13300000
1,1.757010,1.403419,5.405809,2.532024,1,0,0,0,1,2.679409,0,0,12250000
2,2.218232,0.047278,1.421812,0.224410,1,0,1,0,0,1.517692,1,1,12250000
3,1.083624,1.403419,1.421812,0.224410,1,0,1,0,1,2.679409,1,0,12215000
4,1.046726,1.403419,-0.570187,0.224410,1,1,1,0,1,1.517692,0,0,11410000


In [225]:
housing_dataset['guestroom'].value_counts()

guestroom
0    448
1     97
Name: count, dtype: int64

In [226]:
df = pd.DataFrame({
 "A": ['X=0', 'Y=1', 'X=0', 'Z=2', 'Y=1']    
})
df['A'] = pd.Categorical(df['A']).codes

In [227]:
""" Model Evaluation 
Split Data: Divide into training, validation, and test sets (e.g., 70/15/15).
            Main idea: Fair distribution of the dataset into train-test-validation
                       For example, we have 1000 dogs image, 500 cats image
                       Option 1: In training select dogs = 500, cats = 250
                       Option 2: In training select dogs = 300, cats = 300  (Correct)
                       
            training: for training the model
            validation: tracking the performance of the model during training and guiding the hyperparameters i.e. learning rate, epochs
            testing: unseen data for testing the performance
Evaluation Metrics: Select task-specific metrics (e.g., accuracy, F1, RMSE).
"""

' Model Evaluation \nSplit Data: Divide into training, validation, and test sets (e.g., 70/15/15).\n            Main idea: Fair distribution of the dataset into train-test-validation\n                       For example, we have 1000 dogs image, 500 cats image\n                       Option 1: In training select dogs = 500, cats = 250\n                       Option 2: In training select dogs = 300, cats = 300  (Correct)\n                       \n            training: for training the model\n            validation: tracking the performance of the model during training and guiding the hyperparameters i.e. learning rate, epochs\n            testing: unseen data for testing the performance\nEvaluation Metrics: Select task-specific metrics (e.g., accuracy, F1, RMSE).\n'

In [228]:
# Separate input values and target values
X = housing_dataset.drop('price', axis=1).values
y = housing_dataset['price'].values
print(X.shape)
print(y.shape)

(545, 12)
(545,)


In [229]:
"""
THEORETICAL CONCEPT: Train-Validation-Test Split
- Training set: Learn parameters (w, b)
- Validation set: Tune hyperparameters, early stopping
- Test set: Final unbiased evaluation (used only once)
- Use stratified split if classification; here regression → random split
- Use `train_test_split` with fixed random_state for reproducibility
"""
# First split: 80% train+val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_val.shape, y_train_val.shape, X_test.shape, y_test.shape

((436, 12), (436,), (109, 12), (109,))

In [230]:
# Second split: 75% train, 25% val from train_val → 60% train, 20% val overall

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

In [231]:
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 327, Val size: 109, Test size: 109


In [232]:
def model(x, w, b):
    y_pred = np.dot(x, w) + b
    return  y_pred

In [233]:
def cost_function(x, y, w, b):
    """Compute Mean Squared Error"""
    y_pred = model(x, w, b)
    mse = np.mean((y - y_pred) ** 2)
    return mse

In [234]:
def compute_gradient(x, y_true, w, b):
    delta = 1e-9
    cost_1 = cost_function(x, y_true, w, b)
    cost_2 = cost_function(x, y_true, w + delta, b)
    cost_3 = cost_function(x, y_true, w, b + delta)
    dw = (cost_2 - cost_1) / delta
    db = (cost_3 - cost_1) / delta
    return dw, db

In [235]:
"""
THEORETICAL CONCEPT: Model Training
- Initialize weights to small random values or zeros
- Use small learning rate for stability
- Monitor training and validation loss
"""

# Initialize parameters
np.random.seed(142)
w_init = np.random.randn(X_train.shape[1]) * 0.01
b_init = 0.0

w_init

array([ 0.0012974 ,  0.00902362,  0.01005804,  0.0047189 , -0.00326213,
       -0.0026265 , -0.00368877,  0.01478741,  0.01299381, -0.00525526,
       -0.00536705,  0.00454478])

In [236]:
# Hyperparameters
learning_rate = 0.01
epochs = 50000

In [237]:
# Train model
def fit(
        X_train, y_train, w_init, b_init, learning_rate, epochs, X_val=None, y_val=None
):
    w = w_init.copy()
    b = b_init
    train_losses, val_losses = [], []
    
    for i in range(epochs):
        dw, db = compute_gradient(X_train, y_train, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db
    
        if i % 100 == 0:
            train_loss = cost_function(X_train, y_train, w, b)
            train_losses.append(train_loss)
            if X_val is not None and y_val is not None:
                val_loss = cost_function(X_val, y_val, w, b)
                val_losses.append(val_loss)
            else:
                val_losses.append(None)
    return w, b, train_losses , val_losses 

In [238]:
# Train model
w_final, b_final, train_losses, val_losses = fit(
    X_train, y_train, w_init, b_init, learning_rate, epochs, X_val, y_val
)

In [239]:
print("Training completed.")
print(f"Final training loss: {train_losses[-1] // 1e9:.4f}")
print(f"Final validation loss: {val_losses[-1] // 1e9:.4f}")

Training completed.
Final training loss: 1376.0000
Final validation loss: 1408.0000


In [240]:
"""
For larger dataset:
-------------------
seed=42, Training completed.
Final training loss: 1378.0000
Final validation loss: 1411.0000

seed=41, Training completed.
Final training loss: 1394.0000
Final validation loss: 1438.0000

seed=142, Training completed.
Final training loss: 1376.0000
Final validation loss: 1408.0000

Average, training loss = (1378+1394+1376) // 3

For smaller dataset
--------------------
Cross validation
"""

'\nseed=42, Training completed.\nFinal training loss: 1378.0000\nFinal validation loss: 1411.0000\n\nseed=41, Training completed.\nFinal training loss: 1394.0000\nFinal validation loss: 1438.0000\n\nseed=142\n'

In [247]:
"""
THEORETICAL CONCEPT: K-Fold Cross-Validation
- Splits data into K folds
- Train on K-1, validate on 1 → repeat K times
- Average performance → reduces variance in evaluation
- Especially useful with small datasets
""";

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mse_scores = []

print("Performing 5-Fold Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
    X_fold_train, X_fold_val = X_train_val[train_idx], X_train_val[val_idx]
    y_fold_train, y_fold_val = y_train_val[train_idx], y_train_val[val_idx]

    # Re-initialize model
    w_fold = np.random.randn(X_fold_train.shape[1]) * 0.01
    b_fold = 0.0

    # Train
    w_fold, b_fold, _, _ = fit(
        X_fold_train, y_fold_train, w_fold, b_fold, learning_rate, epochs
    )

    # Evaluate
    y_pred_val = X_fold_val @ w_fold + b_fold
    mse = mean_squared_error(y_fold_val, y_pred_val)
    cv_mse_scores.append(mse)
    print(f"  Fold {fold+1} MSE: {mse:.4f}")

print(f"Cross-Validation MSE: {np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}")

Performing 5-Fold Cross-Validation...
  Fold 1 MSE: 1549279376625.6194
  Fold 2 MSE: 1156252253606.9424
  Fold 3 MSE: 1320790065508.4656
  Fold 4 MSE: 1697629464829.7029
  Fold 5 MSE: 1188605629771.4106
Cross-Validation MSE: 1382511358068.4282 ± 209624962729.0055


In [249]:
"""
THEORETICAL CONCEPT: KL-Divergence for Distribution Matching
- Measures how well predicted distribution matches true distribution
- Lower KL → better alignment in predictive uncertainty
"""

def kl_divergence(y_true, y_pred, bins=50):
    hist_true, _ = np.histogram(y_true, bins=bins, density=True)
    hist_pred, _ = np.histogram(y_pred, bins=bins, density=True)
    return entropy(hist_true + 1e-10, hist_pred + 1e-10)

y_train_pred = model(X_train, w_final, b_final)
y_val_pred = model(X_val, w_final, b_final)
y_test_pred = model(X_test, w_final, b_final)

kl_train = kl_divergence(y_train, y_train_pred)
kl_val = kl_divergence(y_val, y_val_pred)
kl_test = kl_divergence(y_test, y_test_pred)

In [250]:
print(f"KL Divergence → Train: {kl_train:.4f}, Val: {kl_val:.4f}, Test: {kl_test:.4f}")

KL Divergence → Train: 0.4826, Val: 1.0474, Test: 0.7094


In [251]:
# Manual learning curve with our model
def plot_learning_curve(X, y, X_val, y_val):
    train_errors, val_errors = [], []
    m = len(X)
    sizes = np.linspace(100, m, 10).astype(int)

    for size in sizes:
        errors_train, errors_val = [], []
        for _ in range(3):  # Average over 3 runs
            idx = np.random.choice(m, size, replace=False)
            X_sub, y_sub = X[idx], y[idx]

            w_sub = np.random.randn(X_sub.shape[1]) * 0.01
            b_sub = 0.0
            w_sub, b_sub, _, _ = fit(X_sub, y_sub, w_sub, b_sub, 0.01, 3000)

            train_err = cost_function(X_sub, y_sub, w_sub, b_sub)
            val_err = cost_function(X_val, y_val, w_sub, b_sub)
            errors_train.append(train_err)
            errors_val.append(val_err)

        train_errors.append(np.mean(errors_train))
        val_errors.append(np.mean(errors_val))
    plt.figure()
    plt.plot(sizes, train_errors, 'o-', label='Training error')
    plt.plot(sizes, val_errors, 'o-', label='Validation error')
    plt.title('Learning Curve: Bias-Variance Diagnosis')
    plt.xlabel('Training Set Size')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)
    plt.show()

In [252]:
"""
THEORETICAL CONCEPT: Learning Curves (Bias-Variance Diagnosis)
- Plot train/validation error vs. training size
- High bias: both errors high and close
- High variance: train error low, val error high (gap)
"""

plot_learning_curve(X_train, y_train, X_val, y_val)

<IPython.core.display.Javascript object>

In [241]:
"""
THEORETICAL CONCEPT: Bias-Variance Tradeoff Summary

Bias: The error due to overly simplistic assumptions in the model, causing consistent underfitting.
Variance: The error due to excessive sensitivity to small fluctuations in the training data, causing overfitting.

Underfitting: Model performs poor in training set, but poorly on test set
Overfitting: Model performs really good in training set, but poorly on test set

Total Error = Bias² + Variance + Irreducible Error

| Case              | Train Error | Val Error | Gap? | Diagnosis         |
|-------------------|-------------|-----------|------|-------------------|
| High Bias         | High        | High      | Small| Underfitting      |
| High Variance     | Low         | High      | Large| Overfitting       |
| Good Fit          | Low         | Low       | Small| Generalization    |

Actions:
- High Bias → Increase model complexity, add features
- High Variance → More data, regularization, reduce features
"""

print("""
BIAS-VARIANCE DIAGNOSTIC SUMMARY:
- If training and validation errors are both high → High Bias (add features/polynomial terms)
- If training error low, validation high → High Variance (regularize, more data)
- Use cross-validation and learning curves to confirm.
""")